# Case Study 3 — RFM Customer Segmentation

Run this notebook in the directory containing `Online_Retail.csv`. Results are generated from the full Kaggle export; no metrics are pre-filled.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
RANDOM_STATE = 42


In [ ]:
raw = pd.read_csv('Online_Retail.csv', encoding='ISO-8859-1')
raw['InvoiceDate'] = pd.to_datetime(raw['InvoiceDate'])
df = raw.dropna(subset=['CustomerID']).copy()
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print(f'Raw rows: {len(raw):,}; valid purchase lines: {len(df):,}; customers: {df.CustomerID.nunique():,}')


In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()
features = rfm[['Recency', 'Frequency', 'Monetary']].copy()
features[['Frequency', 'Monetary']] = np.log1p(features[['Frequency', 'Monetary']])
X = StandardScaler().fit_transform(features)
rfm.describe().round(2)


In [ ]:
scores = {}
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X)
    scores[k] = {'inertia': model.inertia_, 'silhouette': silhouette_score(X, model.labels_)}
pd.DataFrame(scores).T


In [ ]:
K = 4  # selected after reviewing the previous score table and interpretability
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
rfm['KMeansCluster'] = kmeans.fit_predict(X)
profile = rfm.groupby('KMeansCluster')[['Recency','Frequency','Monetary']].agg(['mean','median','count']).round(2)
display(profile)
print('Silhouette:', round(silhouette_score(X, rfm.KMeansCluster), 3))


In [ ]:
dbscan = DBSCAN(eps=0.55, min_samples=12)
rfm['DBSCANCluster'] = dbscan.fit_predict(X)
n_clusters = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
print('DBSCAN clusters:', n_clusters, ' noise share:', round((dbscan.labels_ == -1).mean(), 3))


In [ ]:
pca_xy = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
tsne_xy = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=RANDOM_STATE).fit_transform(X)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for axis, xy, title in zip(ax, [pca_xy, tsne_xy], ['PCA of RFM', 't-SNE of RFM']):
    axis.scatter(xy[:,0], xy[:,1], c=rfm.KMeansCluster, s=8, alpha=.65, cmap='tab10')
    axis.set_title(title)
plt.show()


In [ ]:
means = rfm.groupby('KMeansCluster')[['Recency','Frequency','Monetary']].mean()
ranked = means.sort_values(['Monetary','Frequency','Recency'], ascending=[False,False,True]).index.tolist()
persona = dict(zip(ranked, ['Champions', 'Loyal customers', 'Potential loyalists', 'At-risk / low value']))
rfm['Persona'] = rfm.KMeansCluster.map(persona)
actions = {'Champions':'VIP reward and premium cross-sell','Loyal customers':'Bundle and loyalty offer','Potential loyalists':'Second-purchase incentive','At-risk / low value':'Time-limited win-back message'}
rfm.groupby('Persona')[['Recency','Frequency','Monetary']].mean().round(2).assign(Recommended_action=lambda x: [actions[i] for i in x.index])
